In [1]:
import pandas as pd
import numpy as np
import dblp
from crossref.restful import Works
import requests

In [2]:
crossref_work = Works()

In [18]:
raw_dfs = [
    # "data/cleaned/search-results-dblp-cleaned.csv", 
    # "data/cleaned/search-results-dblp-no-tertiary-cleaned.csv", 
    "data/cleaned/search-results-sch-cleaned.csv",
]

In [4]:
# Prepare DataFrame to store the results
columns = [
    "PaperTitle",
    "DOI",
    "Authors",
    "Abstract",
    "Publisher",
    "DoiUrl",
    "PublicationDate",
    "Conference-Journal",
    "PublicationTypes",
    "SearchString",
    "CitationCount",
    "SearchedFrom",
]

In [5]:
sch_fields = [
    "title",
    "externalIds",
    "authors",
    "abstract",
    "url",
    "publicationDate",
    "fieldsOfStudy",
    "venue",
    "publicationTypes",
    "citationCount",
    "externalIds",
]

In [13]:
def extract_sch(row):
    api_url = f"https://api.semanticscholar.org/graph/v1/paper/{row['ID']}"
    headers = {"Content-Type": "application/json"}
    params = {
        "fields": ",".join(sch_fields),
    }
    response = requests.get(api_url, headers=headers, params=params)
    result = response.json()
    return result

def extract_data(row):
        sch_paper = extract_sch(row)
        doi = sch_paper.get("externalIds", {}).get("ID", None)
        if doi is None and str(row["ID"]).startswith("DOI"):
            doi = row["ID"].split(":")[1]
        try:
            crossref_paper = crossref_work.doi(doi)
        except Exception as e:
            crossref_paper = None

        title = row["PaperTitle"]

        authors = None
        # author name and affiliation
        if crossref_paper is not None:
            authors = crossref_paper.get("author")
        if authors is not None:
            for i in range(len(authors)):
                author = authors[i]
                author_name = author.get("given", "") + " " + author.get("family", "")
                affiliations = author.get("affiliation", [])
                school_names = (
                    [affil.get("name") for affil in affiliations]
                    if affiliations
                    else ["No Affiliation"]
                )
                # Create a new dictionary with only 'name' and 'affiliation'
                authors[i] = {
                    "name": author_name.strip(),
                    "affiliation": school_names,
                }
        else:
            authors = sch_paper.get("authors", [])
            for i in range(len(authors)):
                author = sch_paper["authors"][i]
                sch_paper["authors"][i] = {
                    "name": author.get("name", "No Name"),
                    "affiliation": author.get("affiliation", "No Affiliation"),
                }

        abstract = sch_paper.get("abstract", None)
        sch_url = sch_paper.get("url", None)
        doi_url = f"https://doi.org/{doi}"
        publication_date = sch_paper.get("publicationDate", None)
        fields_of_study = sch_paper.get("fieldsOfStudy", [])
        venue = sch_paper.get("venue", None)

        # publisher
        if crossref_paper is not None:
            publisher = crossref_paper.get("publisher")
        elif doi and "arxiv" in doi.lower():
            publisher = "arXiv"
        else:
            publisher = None

        # paper type
        if crossref_paper is not None:
            paper_type = [crossref_paper.get("type")]
        else:
            paper_type = sch_paper.get("publicationTypes", [])

        citation_count = sch_paper.get("citationCount", None)
        # TODO: paper keywords missing
        # TODO: paper type is conference/journal for arxiv papers
        # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

        new_paper = {
            "PaperTitle": title,
            "DOI": doi,
            "Authors": authors,
            "Abstract": abstract,
            "Publisher": publisher,
            "SemanticScholarUrl": sch_url,
            "DoiUrl": doi_url,
            "PublicationDate": publication_date,
            "FieldOfStudy": fields_of_study,
            "Conference-Journal": venue,
            "PublicationTypes": paper_type,
            "SearchString": row["SearchString"],
            "CitationCount": citation_count,
            "SearchedFrom": row["SearchedFrom"],
        }
        return new_paper

In [19]:
for raw_df_path in raw_dfs:
    raw_df = pd.read_csv(raw_df_path)
    results = []
    
    total_rows = len(raw_df)
    for index, row in raw_df.iterrows():
        print(f"Processing {raw_df_path}: row {index + 1}/{total_rows}...")
        paper = extract_data(row)
        results.append(paper)

    # Creating a DataFrame from the results
    results_df = pd.DataFrame(results, columns=columns)

    # Generating a new file name based on the raw data file name
    new_file_name = raw_df_path.replace(".csv", "-full.csv")

    # Saving to a CSV file
    results_df.to_csv(new_file_name, index=False)

Processing data/cleaned/search-results-sch-cleaned.csv: row 1/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 2/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 3/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 4/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 5/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 6/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 7/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 8/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 9/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 10/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 11/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 12/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 13/2452...
Processing data/cleaned/search-results-sch-cleaned.csv: row 14/2452...
Processing data

In [ ]:
# merge the three new csv, on paper title and doi
